In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np

from src.evaluation import (
    semantic_consistency,
    retrieval_consistency,
)

In [3]:
DATA_DIR = PROJECT_ROOT / "data"

INPUT_PATH = DATA_DIR / "case_embeddings.parquet"

ANONYMIZED_DIR = DATA_DIR / "anonymized_outputs"

OUTPUT_DIR = DATA_DIR / "results"
OUTPUT_DIR.mkdir(exist_ok=True)

CLUSTER_K = [5, 10, 25, 50, 100]

NEIGHBOR_K = [100, 200, 300, 400, 500]

In [ ]:
df = pd.read_parquet(INPUT_PATH)
X = df.to_numpy(dtype=float)
print(f"Loaded {X.shape[0]} embeddings.")

Loaded 1000 embeddings.


In [ ]:
results = []

algorithms = {
    "MDAV-C": "mdav_c_k{}.parquet",
    "RMDAV-M": "rmdav_m_k{}.parquet",
}

for algorithm, pattern in algorithms.items():

    for cluster_k in CLUSTER_K:

        print(f"{algorithm} | k={cluster_k}")

        df_anon = pd.read_parquet(
            ANONYMIZED_DIR / pattern.format(cluster_k)
        )

        X_anon = df_anon.to_numpy(dtype=float)

        sc = semantic_consistency(
            X,
            X_anon,
        )

        for neighbor_k in NEIGHBOR_K:

            rc = retrieval_consistency(
                X,
                X_anon,
                k=neighbor_k,
            )

            results.append(
                {
                    "algorithm": algorithm,
                    "cluster_k": cluster_k,
                    "neighbor_k": neighbor_k,
                    "semantic_consistency": sc,
                    "retrieval_consistency": rc,
                }
            )

MDAV-C | k=5
MDAV-C | k=10
MDAV-C | k=25
MDAV-C | k=50
MDAV-C | k=100
RMDAV-M | k=5
RMDAV-M | k=10
RMDAV-M | k=25
RMDAV-M | k=50
RMDAV-M | k=100


,algorithm,cluster_k,neighbor_k,semantic_consistency,retrieval_consistency
0,MDAV-C,5,100,0.877192,0.433230
1,MDAV-C,5,200,0.877192,0.521080
2,MDAV-C,5,300,0.877192,0.588360
3,MDAV-C,5,400,0.877192,0.649027
4,MDAV-C,5,500,0.877192,0.707102
5,MDAV-C,10,100,0.841316,0.333950
6,MDAV-C,10,200,0.841316,0.430595
7,MDAV-C,10,300,0.841316,0.508677
8,MDAV-C,10,400,0.841316,0.579348
9,MDAV-C,10,500,0.841316,0.648000


In [ ]:
results_df = pd.DataFrame(results)

results_df

In [6]:
results_df.to_csv(
    OUTPUT_DIR / "utility_results.csv",
    index=False,
)

print("Utility evaluation completed successfully.")

Utility evaluation completed successfully.
